1. VS Code에서 01_data_merge.ipynb 파일 생성

scripts/ 폴더 안에 새로운 주피터 노트북 파일을 만드세요.  여기서 데이터를 합치고 정제하는 코드를 작성할 겁니다.

2. 데이터 불러오기 및 헤더 정리
상권 데이터의 헤더가 밀려 있던 부분을 바로잡으며 불러와야 합니다.

In [1]:
import sys
!{sys.executable} -m pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [ ]:
# 1. 지하철 데이터 로드
try:
    subway = pd.read_csv('../data/subway_data.csv', encoding='utf-8')
except:
    subway = pd.read_csv('../data/subway_data.csv', encoding='cp949')

# 2. 상권 데이터 로드 (오류가 났던 부분 수정)
cols = ['상호명', '상권업종소분류명', '시군구명', '행정동명', '경도', '위도']
try:
    # 0xbf 오류 해결을 위해 cp949로 시도
    shop = pd.read_csv('../data/seoul_shop_data.csv', header=1, usecols=cols, encoding='cp949')
except UnicodeDecodeError:
    # 그래도 안 될 경우 utf-8-sig 시도
    shop = pd.read_csv('../data/seoul_shop_data.csv', header=1, usecols=cols, encoding='utf-8-sig')

# 정상적으로 실행되었는지 확인
print("✅ 모든 데이터 로드 성공!")
display(subway.head())
display(shop.head())

✅ 모든 데이터 로드 성공!


,외구간_역_수,역한글명칭,호선명칭,환승역X좌표,환승역Y좌표
0,4128,삼성중앙,9호선(연장),127.053282,37.513011
1,4124,사평,9호선,127.015259,37.504206
2,4121,구반포,9호선,126.987332,37.501364
3,4119,흑석(중앙대입구),9호선,126.963708,37.508770
4,4116,샛강,9호선,126.928422,37.517274


,상호명,상권업종소분류명,시군구명,행정동명,경도,위도
0,60계치킨암사,치킨,강동구,암사2동,127.126859,37.550810
1,성심인력공사,고용 알선업,용산구,남영동,126.972240,37.552803
2,칸토빈,카페,강서구,공항동,126.810493,37.563548
3,까치노래연습장,노래방,노원구,상계5동,127.071218,37.660715
4,금빛주얼리,시계/귀금속 소매업,종로구,종로1.2.3.4가동,126.991861,37.572331


3. 동(洞) 단위 인프라 개수 집계 (핵심!)
정량화 및 가중치 적용을 위해, 각 동네에 편의시설이 몇 개씩 있는지 계산해야 합니다.

In [ ]:
# 예를 들어 '카페'와 '편의점'만 필터링한다면
cafe_count = shop[shop['상권업종소분류명'] == '카페'].groupby('행정동명').size().reset_index(name='카페수')
store_count = shop[shop['상권업종소분류명'] == '편의점'].groupby('행정동명').size().reset_index(name='편의점수')

# 지하철역은 '동' 정보가 없을 수 있으니, 나중에 좌표 기반으로 계산하거나 
# 주소 정보가 있는 파일을 합쳐 '동'별 개수를 구합니다.

4. 통합 마스터 테이블 생성
위에서 구한 개수들을 하나의 테이블로 합칩니다. 이 테이블이 나중에 추천 알고리즘의 입력값이 됩니다.